# MiniTienda - Registro y análisis de ventas
**Lógica de Programación - UIDE**

Este es mi notebook para el desafío MiniTienda: un programa de consola que
mantiene un catálogo de productos, registra ventas, guarda y lee los datos
en un CSV, calcula algunas métricas con NumPy y al final grafica los
ingresos por producto con Matplotlib.

**Estructuras de datos que usé**
- **Tuplas:** cada producto del catálogo es una tupla `(id, nombre, categoria)`.
- **Diccionarios:** `precios` y `stock`, indexados por `id_producto`.
- **Listas:** `catalogo` (lista de tuplas) y `ventas_buffer` (lista de diccionarios, ahí se van guardando las ventas).
- **Pandas:** paso las ventas a un `DataFrame`, uso `groupby` para sacar los ingresos por producto y lo guardo/leo como CSV.
- **NumPy:** `mean`, `std`, `sum` sobre los ingresos y las cantidades.
- **Matplotlib:** el gráfico de barras de ingresos por producto.
- **Control de flujo:** `if/elif/else`, `for`, `while`, `break`, `continue`, `try/except/else/finally`.


In [1]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Librerías importadas correctamente:", pd.__version__, np.__version__)

Librerías importadas correctamente: 3.0.2 2.4.4


## 1) Catálogo (tuplas), precios y stock (diccionarios)

El catálogo lo hice como una **lista de tuplas** `(id, nombre, categoria)`.
Usé una lista (y no una tupla fija) para poder ir agregando productos
nuevos más adelante (eso es el Reto A), pero cada producto individual
sigue siendo una tupla, que no se puede modificar.

In [2]:
catalogo = [
    (1, "Laptop", "Cómputo"),
    (2, "Mouse", "Accesorios"),
    (3, "Teclado", "Accesorios"),
    (4, "Monitor", "Cómputo"),
    (5, "Audífonos", "Accesorios"),
]

precios = {1: 650.00, 2: 15.50, 3: 25.00, 4: 180.00, 5: 35.00}
stock = {1: 10, 2: 50, 3: 40, 4: 15, 5: 30}

ventas_buffer = []
_contador_id_venta = [0]

ARCHIVO_VENTAS = "ventas.csv"
ARCHIVO_LOG = "log.txt"


## 2) Funciones auxiliares (log, búsqueda, catálogo)

In [3]:
def escribir_log(mensaje):
    """Escribe una línea con fecha/hora en log.txt (archivo)."""
    with open(ARCHIVO_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {mensaje}\n")


def buscar_producto(producto_id):
    """Busca un producto en el catálogo (lista de tuplas) por id."""
    for producto in catalogo:
        if producto[0] == producto_id:
            return producto
    return None


def mostrar_catalogo():
    print("\n--- CATÁLOGO DE PRODUCTOS ---")
    print(f"{'ID':<4}{'Nombre':<15}{'Categoría':<15}{'Precio':<10}{'Stock':<6}")
    for producto in catalogo:
        pid, nombre, categoria = producto
        precio = precios.get(pid, 0)
        existencias = stock.get(pid, 0)
        print(f"{pid:<4}{nombre:<15}{categoria:<15}{precio:<10.2f}{existencias:<6}")

## 3) Reto A - Agregar producto nuevo / actualizar precio y stock

In [4]:
def agregar_producto(producto_id, nombre, categoria, precio, stock_inicial):
    """Agrega un producto nuevo al catálogo o actualiza uno existente (Reto A)."""
    existente = buscar_producto(producto_id)
    if existente is None:
        catalogo.append((producto_id, nombre, categoria))
        precios[producto_id] = precio
        stock[producto_id] = stock_inicial
        escribir_log(f"Producto nuevo agregado: id={producto_id}, nombre={nombre}")
        print(f"Producto '{nombre}' agregado al catálogo.")
    else:
        precios[producto_id] = precio
        stock[producto_id] = stock_inicial
        escribir_log(f"Producto actualizado: id={producto_id}, nombre={nombre}")
        print(f"Producto '{nombre}' actualizado (precio/stock).")

## 4) Registrar venta - Reto C (descuento) y Reto D (log de intentos fallidos)

- Si la `cantidad` es 10 o más, se aplica un **5% de descuento** (Reto C, con un `if`).
- Si el `producto_id` no existe en el catálogo, se lanza un error controlado
  **y además queda anotado el intento fallido en `log.txt`** (Reto D).

In [5]:
def registrar_venta(producto_id, cantidad):
    producto = buscar_producto(producto_id)

    if producto is None:
        # Reto D: registrar intento fallido en el log
        escribir_log(f"INTENTO FALLIDO - producto_id inexistente: {producto_id}")
        raise KeyError(f"El producto con id {producto_id} no existe en el catálogo.")

    if cantidad <= 0:
        raise ValueError("La cantidad debe ser mayor a 0.")

    disponible = stock.get(producto_id, 0)
    if cantidad > disponible:
        escribir_log(
            f"INTENTO FALLIDO - stock insuficiente: producto_id={producto_id}, "
            f"pedido={cantidad}, disponible={disponible}"
        )
        raise ValueError(f"Stock insuficiente. Disponible: {disponible}")

    pid, nombre, categoria = producto
    precio_unitario = precios[pid]
    subtotal = precio_unitario * cantidad

    # Reto C: descuento del 5% si la cantidad es >= 10
    descuento_pct = 0.05 if cantidad >= 10 else 0.0
    descuento_valor = subtotal * descuento_pct
    total = subtotal - descuento_valor

    stock[pid] = disponible - cantidad

    _contador_id_venta[0] += 1
    venta = {
        "id_venta": _contador_id_venta[0],
        "producto_id": pid,
        "producto": nombre,
        "categoria": categoria,
        "cantidad": cantidad,
        "precio_unitario": precio_unitario,
        "descuento_pct": descuento_pct,
        "total": round(total, 2),
        "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    ventas_buffer.append(venta)
    escribir_log(f"Venta registrada: {venta}")
    return venta

## 5) Archivos: guardar / leer `ventas.csv` con Pandas (try/except)

In [6]:
def guardar_ventas_csv(nombre_archivo=ARCHIVO_VENTAS):
    if not ventas_buffer:
        print("No hay ventas registradas todavía; no se generó el CSV.")
        return None
    df = pd.DataFrame(ventas_buffer)
    df.to_csv(nombre_archivo, index=False, encoding="utf-8")
    escribir_log(f"CSV guardado: {nombre_archivo} ({len(df)} filas)")
    return df


def leer_ventas_csv(nombre_archivo=ARCHIVO_VENTAS):
    try:
        df = pd.read_csv(nombre_archivo)
        return df
    except FileNotFoundError:
        print(f"El archivo '{nombre_archivo}' no existe todavía. Registra ventas primero.")
        escribir_log(f"ERROR - intento de lectura de archivo inexistente: {nombre_archivo}")
        return None

## 6) Métricas con NumPy (mean, std, sum) - con la división por cero controlada

In [7]:
def calcular_metricas(df=None):
    if df is None:
        df = pd.DataFrame(ventas_buffer)

    if df is None or df.empty:
        print("No hay datos suficientes para calcular métricas.")
        return None

    ingresos = np.array(df["total"], dtype=float)
    cantidades = np.array(df["cantidad"], dtype=float)
    total_ventas = len(cantidades)

    try:
        promedio_unidades = np.sum(cantidades) / total_ventas
    except ZeroDivisionError:
        promedio_unidades = 0.0

    metricas = {
        "ingreso_total": float(np.sum(ingresos)),
        "ingreso_promedio": float(np.mean(ingresos)),
        "ingreso_desviacion_std": float(np.std(ingresos)),
        "unidades_totales": float(np.sum(cantidades)),
        "promedio_unidades_por_venta": float(promedio_unidades),
    }
    return metricas


def mostrar_metricas(metricas):
    if metricas is None:
        return
    print("\n--- MÉTRICAS (NumPy) ---")
    print(f"Ingreso total:            ${metricas['ingreso_total']:.2f}")
    print(f"Ingreso promedio/venta:   ${metricas['ingreso_promedio']:.2f}")
    print(f"Desviación estándar:      ${metricas['ingreso_desviacion_std']:.2f}")
    print(f"Unidades totales vendidas:{metricas['unidades_totales']:.0f}")
    print(f"Promedio unidades/venta:  {metricas['promedio_unidades_por_venta']:.2f}")

## 7) Pandas `groupby` + Matplotlib - gráfico de ingresos por producto (Reto B)

In [8]:
def ingresos_por_producto(df=None):
    if df is None:
        df = pd.DataFrame(ventas_buffer)
    if df is None or df.empty:
        return None
    resumen = df.groupby("producto")["total"].sum().sort_values(ascending=False)
    return resumen


def graficar_ingresos(df=None, guardar_png=False, nombre_png="ingresos.png"):
    resumen = ingresos_por_producto(df)
    if resumen is None:
        print("No hay ventas para graficar.")
        return

    plt.figure(figsize=(8, 5))
    resumen.plot(kind="bar", color="#4C72B0")
    plt.title("Ingresos por producto - MiniTienda")
    plt.xlabel("Producto")
    plt.ylabel("Ingresos ($)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    if guardar_png:
        plt.savefig(nombre_png)  # Reto B: exportar a PNG
        print(f"Gráfico exportado como '{nombre_png}'.")
        escribir_log(f"Gráfico exportado a PNG: {nombre_png}")

    plt.show()

## 8) Menú de consola (while, if/elif/else, for, break, continue, try/except/else/finally)

> Esta celda define el menú tal como correría en la consola de verdad
> (usa `input()`). Al ejecutar el notebook completo esta celda no se
> corre sola porque necesita que uno escriba algo; las pruebas están en
> la sección 9 de abajo, que simulan todo sin pedir nada por teclado.
> Si quieres probar el menú interactivo, descomenta la última línea
> (`menu()`) y corre la celda.

In [9]:
def pedir_entero(mensaje):
    while True:
        try:
            valor = int(input(mensaje))
        except ValueError:
            print("Entrada inválida. Debe ingresar un número entero.")
            continue
        else:
            return valor


def menu():
    opciones_validas = {"1", "2", "3", "4", "5", "6", "7", "0"}

    while True:
        print("\n===== MENÚ MINITIENDA =====")
        print("1) Ver catálogo")
        print("2) Registrar venta")
        print("3) Guardar ventas en CSV")
        print("4) Leer ventas desde CSV")
        print("5) Calcular métricas")
        print("6) Exportar gráfico a PNG")
        print("7) Agregar / actualizar producto")
        print("0) Salir")

        opcion = input("Seleccione una opción: ").strip()

        if opcion not in opciones_validas:
            print("Opción no reconocida, intente de nuevo.")
            continue

        if opcion == "0":
            print("Guardando datos antes de salir...")
            guardar_ventas_csv()
            print("¡Hasta luego!")
            break

        elif opcion == "1":
            mostrar_catalogo()

        elif opcion == "2":
            try:
                pid = pedir_entero("ID del producto: ")
                cant = pedir_entero("Cantidad: ")
                venta = registrar_venta(pid, cant)
            except KeyError as e:
                print(f"Error: {e}")
            except ValueError as e:
                print(f"Error: {e}")
            except Exception as e:
                print(f"Error inesperado: {e}")
            else:
                print(f"Venta registrada con éxito: {venta}")
            finally:
                print("Fin del intento de registro de venta.\n")

        elif opcion == "3":
            guardar_ventas_csv()

        elif opcion == "4":
            df = leer_ventas_csv()
            if df is not None:
                print(df)

        elif opcion == "5":
            df = leer_ventas_csv()
            metricas = calcular_metricas(df) if df is not None else calcular_metricas()
            mostrar_metricas(metricas)

        elif opcion == "6":
            df = leer_ventas_csv()
            graficar_ingresos(df, guardar_png=True)

        elif opcion == "7":
            try:
                pid = pedir_entero("ID del producto nuevo/existente: ")
                nombre = input("Nombre: ").strip()
                categoria = input("Categoría: ").strip()
                precio = float(input("Precio: "))
                stock_inicial = pedir_entero("Stock inicial: ")
                agregar_producto(pid, nombre, categoria, precio, stock_inicial)
            except ValueError as e:
                print(f"Error: {e}")

# Para usar el menú interactivo, descomentar la siguiente línea y ejecutar la celda:
# menu()

## 9) Celdas de prueba

Acá simulo el uso completo del programa **sin necesidad de escribir nada
a mano**: registro varias ventas, provoco a propósito un intento fallido
(para el Reto D) y una venta con descuento (Reto C), y genero todos los
archivos que pide la consigna.

In [10]:
print(">>> Catálogo inicial")
mostrar_catalogo()

>>> Catálogo inicial

--- CATÁLOGO DE PRODUCTOS ---
ID  Nombre         Categoría      Precio    Stock 
1   Laptop         Cómputo        650.00    10    
2   Mouse          Accesorios     15.50     50    
3   Teclado        Accesorios     25.00     40    
4   Monitor        Cómputo        180.00    15    
5   Audífonos      Accesorios     35.00     30    


In [11]:
print(">>> Reto A: agregar producto nuevo")
agregar_producto(6, "Webcam", "Accesorios", 45.00, 20)
mostrar_catalogo()

>>> Reto A: agregar producto nuevo
Producto 'Webcam' agregado al catálogo.

--- CATÁLOGO DE PRODUCTOS ---
ID  Nombre         Categoría      Precio    Stock 
1   Laptop         Cómputo        650.00    10    
2   Mouse          Accesorios     15.50     50    
3   Teclado        Accesorios     25.00     40    
4   Monitor        Cómputo        180.00    15    
5   Audífonos      Accesorios     35.00     30    
6   Webcam         Accesorios     45.00     20    


In [12]:
print(">>> Registrando ventas de prueba (incluye descuento Reto C)")
ventas_prueba = [
    (1, 1), (2, 5), (3, 12), (4, 2), (5, 3),
    (2, 10), (1, 2), (6, 4), (3, 3), (4, 1),
    (5, 15), (2, 3),
]

for pid, cantidad in ventas_prueba:
    try:
        venta = registrar_venta(pid, cantidad)
    except (KeyError, ValueError) as e:
        print(f"  [Error controlado] producto_id={pid}, cantidad={cantidad} -> {e}")
    else:
        print(f"  Venta OK: id_venta={venta['id_venta']}, producto={venta['producto']}, total=${venta['total']}")

print(f"\nTotal de ventas registradas: {len(ventas_buffer)}")

>>> Registrando ventas de prueba (incluye descuento Reto C)
  Venta OK: id_venta=1, producto=Laptop, total=$650.0
  Venta OK: id_venta=2, producto=Mouse, total=$77.5
  Venta OK: id_venta=3, producto=Teclado, total=$285.0
  Venta OK: id_venta=4, producto=Monitor, total=$360.0
  Venta OK: id_venta=5, producto=Audífonos, total=$105.0
  Venta OK: id_venta=6, producto=Mouse, total=$147.25
  Venta OK: id_venta=7, producto=Laptop, total=$1300.0
  Venta OK: id_venta=8, producto=Webcam, total=$180.0
  Venta OK: id_venta=9, producto=Teclado, total=$75.0
  Venta OK: id_venta=10, producto=Monitor, total=$180.0
  Venta OK: id_venta=11, producto=Audífonos, total=$498.75
  Venta OK: id_venta=12, producto=Mouse, total=$46.5

Total de ventas registradas: 12


In [13]:
print(">>> Reto D: venta con producto_id inexistente (999) -> se registra en log.txt")
try:
    registrar_venta(999, 1)
except KeyError as e:
    print(f"[Error controlado] {e}")

print("\n>>> Cantidad inválida (0 unidades)")
try:
    registrar_venta(1, 0)
except ValueError as e:
    print(f"[Error controlado] {e}")

>>> Reto D: venta con producto_id inexistente (999) -> se registra en log.txt
[Error controlado] 'El producto con id 999 no existe en el catálogo.'

>>> Cantidad inválida (0 unidades)
[Error controlado] La cantidad debe ser mayor a 0.


In [14]:
print(">>> Guardando ventas.csv con Pandas")
df = guardar_ventas_csv()
df

>>> Guardando ventas.csv con Pandas


,id_venta,producto_id,producto,categoria,cantidad,precio_unitario,descuento_pct,total,fecha
0,1,1,Laptop,Cómputo,1,650.0,0.00,650.00,2026-08-20 00:01:15
1,2,2,Mouse,Accesorios,5,15.5,0.00,77.50,2026-08-20 00:01:15
2,3,3,Teclado,Accesorios,12,25.0,0.05,285.00,2026-08-20 00:01:15
3,4,4,Monitor,Cómputo,2,180.0,0.00,360.00,2026-08-20 00:01:15
4,5,5,Audífonos,Accesorios,3,35.0,0.00,105.00,2026-08-20 00:01:15
5,6,2,Mouse,Accesorios,10,15.5,0.05,147.25,2026-08-20 00:01:15
6,7,1,Laptop,Cómputo,2,650.0,0.00,1300.00,2026-08-20 00:01:15
7,8,6,Webcam,Accesorios,4,45.0,0.00,180.00,2026-08-20 00:01:15
8,9,3,Teclado,Accesorios,3,25.0,0.00,75.00,2026-08-20 00:01:15
9,10,4,Monitor,Cómputo,1,180.0,0.00,180.00,2026-08-20 00:01:15


In [15]:
print(">>> Leyendo ventas.csv de vuelta")
df_leido = leer_ventas_csv()
df_leido.head(12)

>>> Leyendo ventas.csv de vuelta


,id_venta,producto_id,producto,categoria,cantidad,precio_unitario,descuento_pct,total,fecha
0,1,1,Laptop,Cómputo,1,650.0,0.00,650.00,2026-08-20 00:01:15
1,2,2,Mouse,Accesorios,5,15.5,0.00,77.50,2026-08-20 00:01:15
2,3,3,Teclado,Accesorios,12,25.0,0.05,285.00,2026-08-20 00:01:15
3,4,4,Monitor,Cómputo,2,180.0,0.00,360.00,2026-08-20 00:01:15
4,5,5,Audífonos,Accesorios,3,35.0,0.00,105.00,2026-08-20 00:01:15
5,6,2,Mouse,Accesorios,10,15.5,0.05,147.25,2026-08-20 00:01:15
6,7,1,Laptop,Cómputo,2,650.0,0.00,1300.00,2026-08-20 00:01:15
7,8,6,Webcam,Accesorios,4,45.0,0.00,180.00,2026-08-20 00:01:15
8,9,3,Teclado,Accesorios,3,25.0,0.00,75.00,2026-08-20 00:01:15
9,10,4,Monitor,Cómputo,1,180.0,0.00,180.00,2026-08-20 00:01:15


In [16]:
print(">>> Prueba de manejo de error: archivo inexistente")
resultado = leer_ventas_csv("no_existe.csv")
print("Resultado:", resultado)

>>> Prueba de manejo de error: archivo inexistente
El archivo 'no_existe.csv' no existe todavía. Registra ventas primero.
Resultado: None


In [17]:
print(">>> Ingresos por producto (Pandas groupby)")
resumen = ingresos_por_producto(df_leido)
resumen

>>> Ingresos por producto (Pandas groupby)


producto
Laptop       1950.00
Audífonos     603.75
Monitor       540.00
Teclado       360.00
Mouse         271.25
Webcam        180.00
Name: total, dtype: float64

In [18]:
print(">>> Métricas con NumPy")
metricas = calcular_metricas(df_leido)
mostrar_metricas(metricas)
metricas

>>> Métricas con NumPy

--- MÉTRICAS (NumPy) ---
Ingreso total:            $3905.00
Ingreso promedio/venta:   $325.42
Desviación estándar:      $343.10
Unidades totales vendidas:61
Promedio unidades/venta:  5.08


{'ingreso_total': 3905.0,
 'ingreso_promedio': 325.4166666666667,
 'ingreso_desviacion_std': 343.1017270998339,
 'unidades_totales': 61.0,
 'promedio_unidades_por_venta': 5.083333333333333}

In [19]:
print(">>> Gráfico de ingresos por producto (Reto B: exportado a ingresos.png)")
graficar_ingresos(df_leido, guardar_png=True, nombre_png="ingresos.png")

>>> Gráfico de ingresos por producto (Reto B: exportado a ingresos.png)


Gráfico exportado como 'ingresos.png'.


In [20]:
print(">>> Contenido de log.txt")
with open("log.txt", "r", encoding="utf-8") as f:
    print(f.read())

>>> Contenido de log.txt


[2026-08-20 00:01:15] Producto nuevo agregado: id=6, nombre=Webcam
[2026-08-20 00:01:15] Venta registrada: {'id_venta': 1, 'producto_id': 1, 'producto': 'Laptop', 'categoria': 'Cómputo', 'cantidad': 1, 'precio_unitario': 650.0, 'descuento_pct': 0.0, 'total': 650.0, 'fecha': '2026-08-20 00:01:15'}
[2026-08-20 00:01:15] Venta registrada: {'id_venta': 2, 'producto_id': 2, 'producto': 'Mouse', 'categoria': 'Accesorios', 'cantidad': 5, 'precio_unitario': 15.5, 'descuento_pct': 0.0, 'total': 77.5, 'fecha': '2026-08-20 00:01:15'}
[2026-08-20 00:01:15] Venta registrada: {'id_venta': 3, 'producto_id': 3, 'producto': 'Teclado', 'categoria': 'Accesorios', 'cantidad': 12, 'precio_unitario': 25.0, 'descuento_pct': 0.05, 'total': 285.0, 'fecha': '2026-08-20 00:01:15'}
[2026-08-20 00:01:15] Venta registrada: {'id_venta': 4, 'producto_id': 4, 'producto': 'Monitor', 'categoria': 'Cómputo', 'cantidad': 2, 'precio_unitario': 180.0, 'descuento_pct': 0.0, 'total': 360.0, 'fecha': '2026-08-20 00:01:15'}
[2

## 10) Preguntas de la asignación

**¿Qué parte la hice con Pandas? ¿Qué parte con NumPy?**

Con Pandas armé el `DataFrame` a partir de la lista de ventas, guardé y
leí el archivo `ventas.csv`, y agrupé las ventas por producto con
`groupby("producto")["total"].sum()` para sacar los ingresos que después
se grafican. Con NumPy hice los cálculos numéricos: `np.sum`, `np.mean` y
`np.std` sobre los arreglos de ingresos y cantidades, para sacar el
ingreso total, el promedio por venta y la desviación estándar.

**¿Dónde usé try/except y por qué?**

Lo usé en tres partes: en `pedir_entero()`, para capturar el error si el
usuario escribe algo que no es un número; en el menú, alrededor de
`registrar_venta()`, para capturar cuando el producto no existe o la
cantidad/stock no son válidos, y así el programa no se cae; y en
`leer_ventas_csv()`, para capturar el error si el archivo `ventas.csv`
todavía no existe. En el registro de venta usé el try/except/else/finally
completo: el `else` corre solo si la venta se guardó bien, y el `finally`
siempre imprime un mensaje al final, haya salido bien o mal.

**¿Qué estructuras son tuplas, listas y diccionarios en el código?**

- **Tuplas:** cada producto individual, `(id, nombre, categoria)`.
- **Listas:** `catalogo` (la lista de tuplas de productos) y `ventas_buffer`
  (la lista de diccionarios donde se van guardando las ventas).
- **Diccionarios:** `precios` y `stock` (con el `id_producto` como clave),
  y cada venta guardada dentro de `ventas_buffer` también es un diccionario.
